# Two-Tower Model — complementary products

**The goal of this notebook is to build a two-tower neural network (TTN) that
finds complementary products** — given an item a user is looking at, retrieve
the items that are bought *alongside* it rather than the items most similar to
it. A phone case complements a phone; another phone does not.

The approach follows **[Suggest, complement, inspire: story of Two Tower
recommendations at Allegro.com](https://arxiv.org/html/2508.03702v1)**
(Osowska-Kurczab, Nazarko, Marzec, Wojciechowska & Kremeňová, RecSys '25),
whose Complementary-TT model is the architecture this work is based on.

## Both towers describe items

This is the part that differs from the classic user/item two-tower setup, and
it shapes every column decision below: **the query tower and the candidate
tower both consume item information.** Neither tower is a user tower.

```
   query ITEM features                    candidate ITEM features
        │                                          │
   ┌────▼────┐                                ┌────▼────┐
   │  QUERY  │  product encoder               │CANDIDATE│  product encoder
   │  TOWER  │  (+ target category)           │  TOWER  │
   └────┬────┘                                └────┬────┘
        │                                          │
   q ∈ ℝ^d  ──────────  score = q · c  ──────────  c ∈ ℝ^d
```

Both towers share the same *architecture* — the paper's "Product Encoder":
each item attribute goes through its own embedding table, the vectors are
concatenated, passed through an MLP and L2-normalised. In the paper the query
tower is the only one modified for the complementary task: the query product
embedding is concatenated with a **target category embedding** drawn from a
one-to-many complementary-category mapping, while the candidate tower stays a
plain product encoder.

That mapping is what `complementary_cats_pairs/` produces —
`data/complementary_categories.pkl`, source category path → target category path,
scored by support and lift. §1 loads it. The co-purchase pairs that supply the
training positives come from the same package's `pairs.ipynb`.

Because both sides are items, per-user history is not a tower input at all and
this notebook does not load it. A user's history still shapes the data — it is
what defines which items count as co-purchased — but that work happens upstream,
in `complementary_cats_pairs/pairs.ipynb`.

## What this notebook covers

It loads the tables the model needs and turns them into training pairs. §1
reads everything; §4 produces `tower_pairs`, one row per directed
(query item, candidate item) example:

| Column | |
| --- | --- |
| `asin_query`, `query_cat_2/3/4` | the item being looked at |
| `asin_target`, `target_cat_2/3/4` | an item bought alongside it, whose category the mapping licenses |

The train/test split is **not** applied here. `co_purchase_pairs.pkl` is
already built from interactions before `date_threshold` (see
`ttn/constants.json`), so the positives in §4 are leak-free by construction.
Model definition and training come next.


In [83]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# data/ lives at the repo root, one level up from this ttn/ folder
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the datasets

Everything this notebook reads, in one place. Nothing below this section opens
a file — every later cell transforms tables that are already in memory.

| Dataset | Grain | What it is |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the interaction log — who bought what, when |
| `df_features.pkl` | one row per `asin` | extracted item attributes: `cat_*`, `brand`, `title_cleaned`, the per-field columns and their parsed measures |
| `co_purchase_pairs.pkl` | one row per item pair | items the same user bought within 90 days, before the cutoff — the **training positives** |
| `complementary_categories.pkl` | one row per directed category pair | which category buys into which, filtered by support and lift — the **complementary mapping** |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin` | the *unfiltered* catalogue; describes 150,826 items `df_features` does not |

`asin` and `reviewerID` are pinned to `str` throughout so ids with leading
zeros (e.g. `0560467893`) survive the read.

Only five of the catalogue's fifteen columns are read. `df_features` already
carries every field the catalogue has — the catalogue's value here is
**coverage**, not extra columns: it describes 28,537 reviewed asins that have
no `df_features` row, every one of them in a `cat_3` with no extraction schema.
Reading all fifteen columns of a 2.1 GB file to use four of them is waste.

All five together peak at about **4.4 GB** of RAM.


In [84]:
# --- 1. Interactions: one row per review ----------------------------------
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

# --- 2. Item features: one row per asin, the extracted attributes ---------
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")

# --- 3. Co-purchase pairs: the training positives -------------------------
co_pairs = pd.read_pickle(DATA_DIR / "co_purchase_pairs.pkl")

# --- 4. The complementary category mapping --------------------------------
comp_cat = pd.read_pickle(DATA_DIR / "complementary_categories.pkl")

# --- 5. The unfiltered catalogue: coverage for items df_features lacks ----
meta_catalogue = pd.read_csv(
    DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    usecols=["asin", "category", "title", "brand", "price"],
    dtype={"asin": str},
    low_memory=False,
)

for name, frame in [
    ("df_reviews", df_reviews), ("df_features", df_features),
    ("co_pairs", co_pairs), ("comp_cat", comp_cat),
    ("meta_catalogue", meta_catalogue),
]:
    print(f"{name:<18} {str(frame.shape):>18}")

print(f"\nunique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
print(f"items described by df_features : {df_features['asin'].nunique():,}")
print(f"items described by the catalogue: {meta_catalogue['asin'].nunique():,}")
df_reviews.head(5)

df_reviews              (6898955, 11)
df_features             (1134566, 92)
co_pairs                (10595885, 2)
comp_cat                   (5845, 11)
meta_catalogue           (1300540, 5)

unique users: 777,242 | unique items: 189,172
items described by df_features : 1,134,566
items described by the catalogue: 1,285,392


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


## 2. Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [85]:
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

1,134,566 rows x 92 columns — 1 failure(s), 0 warning(s)


,check,level,subject,detail
0,columns,FAIL,also_buy,present in the table but not in the contract


## 3. Clean the item category path

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket.

This runs before anything else because the complementary mapping's `cat_4`
values are folded the same way. Joining §4 on the raw `cat_4` would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [86]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


## 4. Preparing the tower data

The first stage of turning the tables above into training examples. Three
steps, no modelling yet — the query/target *roles* are assigned here, but what
each tower does with them comes later.

1. **Join the categories onto both ends.** `co_purchase_pairs.pkl` is just two
   asins; `df_features` supplies each one's category path. Inner join on both
   sides, so a pair survives only if both items have features. Result: 8
   columns — two asins and three category levels each.
2. **Inner join onto the mapping, in order.** `asinA`'s three categories against
   the mapping's *first* three (`src_*`), `asinB`'s against the *last* three
   (`dst_*`). A row survives only where that exact directed category relation
   exists. Here `asinA` is the source, so it becomes the **query**.
3. **Inner join again, reversed.** `asinA`'s categories against `dst_*` and
   `asinB`'s against `src_*`. Now `asinB` is the source, so `asinB` becomes the
   **query** and `asinA` the target.

Then concatenate. Both tables are relabelled so the source side is always
`asin_query` / `query_cat_*` and the target side always `asin_target` /
`target_cat_*`, which is what makes the concat meaningful — otherwise the
query would sit in a different column in each half.

A pair licensed in both directions appears in both tables. That is not
duplication: `X → Y` and `Y → X` are two different training examples, and
which item is the query differs between them.

`edges`, `support` and `lift` are dropped. They did their job when the mapping
was filtered; the model does not consume them.

**`cat_4_clean`, not `cat_4`.** The mapping's `cat_4` values are folded through
`category_taxonomy.json`, and §2 already computes that fold (identical to the
package's `fold_cat_4`). An inner join on the raw column would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [87]:
# --- Step 1: the pair, plus the category path of each end ------------------
CAT_LEVELS = ["cat_2", "cat_3", "cat_4_clean"]     # the mapping's three levels

item_cats = df_features[["asin"] + CAT_LEVELS]

pair_cats = (
    co_pairs
    .assign(asinA=co_pairs["asinA"].astype(str), asinB=co_pairs["asinB"].astype(str))
    .merge(item_cats.add_prefix("a_"), left_on="asinA", right_on="a_asin", how="inner")
    .merge(item_cats.add_prefix("b_"), left_on="asinB", right_on="b_asin", how="inner")
    .drop(columns=["a_asin", "b_asin"])
)

print(f"co_purchase pairs            : {len(co_pairs):>12,}")
print(f"after joining features (both) : {len(pair_cats):>12,} "
      f"({len(pair_cats) / len(co_pairs):.1%})")
print(f"dropped, an asin has no features: {len(co_pairs) - len(pair_cats):>10,}")
print(f"\ncolumns ({pair_cats.shape[1]}): {list(pair_cats.columns)}")
pair_cats.head(5)

co_purchase pairs            :   10,595,885
after joining features (both) :    7,729,936 (73.0%)
dropped, an asin has no features:  2,865,949

columns (8): ['asinA', 'asinB', 'a_cat_2', 'a_cat_3', 'a_cat_4_clean', 'b_cat_2', 'b_cat_3', 'b_cat_4_clean']


,asinA,asinB,a_cat_2,a_cat_3,a_cat_4_clean,b_cat_2,b_cat_3,b_cat_4_clean
0,0560467893,B001E95R0O,Home Dcor,Home Dcor Accents,Corner Shelves,Furniture,Accent Furniture,Storage Trunks
1,0560467893,B001F7SGHQ,Home Dcor,Home Dcor Accents,Corner Shelves,Kitchen & Dining,Kitchen Utensils & Gadgets,Bar & Wine Tools
2,0560467893,B004RAL4H2,Home Dcor,Home Dcor Accents,Corner Shelves,Furniture,Bedroom Furniture,"Beds, Frames & Bases"
3,0560467893,B005C7SRLK,Home Dcor,Home Dcor Accents,Corner Shelves,Kitchen & Dining,Dining & Entertaining,Serveware
4,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers


In [88]:
# The mapping's column names, from the package that built it — ROOT went on
# sys.path in the imports cell.
from complementary_cats_pairs import DST_COLS, SRC_COLS

# --- Steps 2 & 3: the two inner joins, then concatenate --------------------
A_CATS = [f"a_{c}" for c in CAT_LEVELS]
B_CATS = [f"b_{c}" for c in CAT_LEVELS]

QUERY_CATS = ["query_cat_2", "query_cat_3", "query_cat_4"]
TARGET_CATS = ["target_cat_2", "target_cat_3", "target_cat_4"]
FINAL_COLS = ["asin_query", "asin_target"] + QUERY_CATS + TARGET_CATS

# Only the six category columns are needed; the metrics are not model inputs.
mapping = comp_cat[SRC_COLS + DST_COLS].astype(str)


def directed_pairs(query_asin, query_cats, target_asin, target_cats):
    """Keep pairs whose (query path -> target path) is in the mapping.

    An inner join of `pair_cats` onto the mapping, with the named side lined up
    against `src_*` and the other against `dst_*`, then relabelled so the
    source side is always the query.
    """
    out = pair_cats.merge(
        mapping,
        left_on=query_cats + target_cats,
        right_on=SRC_COLS + DST_COLS,
        how="inner",
    )
    return out.rename(columns=dict(
        [(query_asin, "asin_query"), (target_asin, "asin_target")]
        + list(zip(query_cats, QUERY_CATS))
        + list(zip(target_cats, TARGET_CATS))
    ))[FINAL_COLS]


table_1 = directed_pairs("asinA", A_CATS, "asinB", B_CATS)   # asinA is source
table_2 = directed_pairs("asinB", B_CATS, "asinA", A_CATS)   # asinB is source

tower_pairs = pd.concat([table_1, table_2], ignore_index=True)

# One shared category vocabulary per level, applied to BOTH sides.
# Casting each column independently would give each its own category list, and
# that breaks two things: `query_cat_3 == target_cat_3` raises "Categoricals can
# only be compared if 'categories' are the same", and -- the real hazard -- the
# .cat.codes disagree, so the same category would index a different row of a
# shared embedding table depending on which tower it fed.
for query_col, target_col in zip(QUERY_CATS, TARGET_CATS):
    level = pd.CategoricalDtype(
        sorted(set(tower_pairs[query_col]) | set(tower_pairs[target_col])))
    tower_pairs[query_col] = tower_pairs[query_col].astype(level)
    tower_pairs[target_col] = tower_pairs[target_col].astype(level)

print(f"table 1 (asinA -> asinB) : {len(table_1):>12,}")
print(f"table 2 (asinB -> asinA) : {len(table_2):>12,}")
print(f"tower_pairs (concat)     : {len(tower_pairs):>12,}")
print(f"\ncolumns ({tower_pairs.shape[1]}): {list(tower_pairs.columns)}")
print(f"distinct query items   : {tower_pairs['asin_query'].nunique():,}")
print(f"distinct target items  : {tower_pairs['asin_target'].nunique():,}")
print(f"memory                 : "
      f"{tower_pairs.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

# Both sides of a level now share one vocabulary, so they compare directly and
# their codes mean the same thing in either tower.
same_cat_3 = (tower_pairs["query_cat_3"] == tower_pairs["target_cat_3"])
print(f"\nquery and target in the same cat_3 : {same_cat_3.sum():,} "
      f"({same_cat_3.mean():.1%})")
tower_pairs.head(10)

table 1 (asinA -> asinB) :    1,458,143
table 2 (asinB -> asinA) :    1,458,492
tower_pairs (concat)     :    2,916,635

columns (8): ['asin_query', 'asin_target', 'query_cat_2', 'query_cat_3', 'query_cat_4', 'target_cat_2', 'target_cat_3', 'target_cat_4']
distinct query items   : 145,184
distinct target items  : 145,077
memory                 : 368 MB

query and target in the same cat_3 : 1,660,199 (56.9%)


,asin_query,asin_target,query_cat_2,query_cat_3,query_cat_4,target_cat_2,target_cat_3,target_cat_4
0,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
1,0560467893,B0150ZOXEI,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
2,0681795107,B000Z4ETF8,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware
3,0681795107,B003ZYGQVK,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Cookware,Canning
4,0681795107,B00X5ETKU4,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers
5,0768205921,B001JYMAA4,Home Dcor,Clocks,Missing,Kitchen & Dining,Storage & Organization,Food Storage
6,0768205921,B0054R3PVU,Home Dcor,Clocks,Missing,Home Dcor,Clocks,Alarm Clocks
7,0768205921,B00L0MIB8K,Home Dcor,Clocks,Missing,Home Dcor,Clocks,Wall Clocks
8,1574893122,B002HPNDCS,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing
9,1574893122,B00BSF5S7G,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing


## 5. Fold rare brands

`brand_clean` folds brands carried by ten or fewer items into `other_brands`,
taking 98,532 brands down to 12,747.

Nothing in this notebook consumes it yet — it is item-side preprocessing for
the product encoder, which reads `df_features` in both towers. Brand is a
natural encoder input alongside title, price and category, and an embedding
table needs the long tail bucketed: a brand on three items would otherwise get
a vector trained by a handful of gradient updates.

The pipeline also produces this now (`clean_brand` is Filter 5), so once you
re-extract, this cell recomputes what the pickle already carries.


In [89]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items. A brand
# on three items would get an embedding trained by a handful of gradient
# updates; bucketing those into one `other_brands` symbol is more honest than
# pretending each has a learned vector.
MIN_ITEMS = 10
OTHER = "other_brands"

# Count on brand_norm where the pipeline produced it, so "3d rose" and "3drose"
# are not counted separately and pushed under the threshold by a split spelling.
source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
brand_counts = df_features.groupby(source)["asin"].nunique()

kept = brand_counts[brand_counts > MIN_ITEMS].index
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept) | df_features[source].isna(),
    OTHER,
)

n_before = df_features[source].nunique()
n_after = df_features["brand_clean"].nunique()
n_missing = df_features[source].isna().sum()
print(f"counted on : {source}")
print(f"brands     : {n_before:,} -> {n_after:,} "
      f"(kept {len(kept):,} with > {MIN_ITEMS} items, rest -> {OTHER!r})")
print(f"items      : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{n_missing / len(df_features):.1%} missing (left as NaN)")
print()
print(df_features["brand_clean"].value_counts().head(10))

counted on : brand
brands     : 98,532 -> 12,747 (kept 12,746 with > 10 items, rest -> 'other_brands')
items      : 16.5% in other_brands, 5.7% missing (left as NaN)

brand_clean
other_brands     187662
3dRose             8569
CafePress          7489
Disney             6249
Unknown            5594
Hallmark           4720
Generic            4500
Department 56      3416
Safavieh           3227
Kurt Adler         3124
Name: count, dtype: int64


## 6. Attach the item attributes both towers read

`tower_pairs` so far carries only asins and category paths. The product encoder
needs the item's own content, so each side gains seven more columns. Names are
lowercase throughout, matching the columns already there:

| Column | Source | Coverage on paired items |
| --- | --- | --- |
| `*_title_cleaned` | `title_cleaned` — lowercased, marketing phrases and stopwords stripped | 100% |
| `*_price` | `price`, parsed to a number | 75% |
| `*_brand_clean` | `brand_clean` from §5 — rare brands folded into `other_brands` | 99% |
| `*_product_type` | `Product_Type` | 76.5% |
| `*_material` | `Material` | 72.5% |
| `*_features` | `Features` | 61.7% |
| `*_color` | `Color` | 54.2% |

The first three are catalogue fields and nearly complete. The last four come out
of the LLM extraction and are present only where the extractor found something —
`Color` is missing on close to half the items. §7 gives every gap an explicit
`Missing` level, so they arrive as usable inputs rather than holes, but the
sparsity is real and worth weighing before putting `color` in the encoder.

**`price` needs parsing, not just carrying.** It is stored as a string
(`'$37.00'`), and 7,032 of its non-null values are not prices at all but
scraped CSS — hundreds of characters of `.a-box-inner{background-color:#fff}…`.
Stripping the currency symbol and coercing handles both at once: real prices
become floats, the junk becomes `NaN`.

Attached by `.map`, not `merge`, so the cell can be re-run safely — a merge
would collide with the columns it added last time and rename them `_x`/`_y`.
No row is lost either way: every asin here came from `df_features` to begin
with.

Every categorical gets **one shared vocabulary across the two sides**, as the
category levels did in §4: a value must index the same row of a shared
embedding table whichever tower it feeds.


In [90]:
# --- Item attributes, one row per asin ------------------------------------
# Source column in df_features -> the name it takes in tower_pairs.
ITEM_ATTRS = {
    "title_cleaned": "title_cleaned",
    "price": "price",
    "brand_clean": "brand_clean",
    "Color": "color",
    "Features": "features",
    "Material": "material",
    "Product_Type": "product_type",
}
CATEGORICAL_ITEM_ATTRS = ["title_cleaned", "brand_clean",
                          "color", "features", "material", "product_type"]

item_attrs = (df_features[["asin"] + list(ITEM_ATTRS)]
              .rename(columns=ITEM_ATTRS))

# '$37.00' -> 37.00, and the scraped-CSS values -> NaN
item_attrs["price"] = pd.to_numeric(
    item_attrs["price"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce",
)
print(f"price parsed : {item_attrs['price'].notna().sum():,} of "
      f"{len(item_attrs):,} items ({item_attrs['price'].notna().mean():.1%}) | "
      f"median ${item_attrs['price'].median():,.2f}")

# --- Attach to both sides --------------------------------------------------
# Assigned by `.map` rather than merged: assignment overwrites, so re-running
# this cell on an already-enriched `tower_pairs` is safe. A merge would collide
# with the columns it added last time and rename them to _x / _y, and the
# lookups below would then fail with KeyError.
attrs_by_asin = item_attrs.set_index("asin")
for side in ("query", "target"):
    asin = tower_pairs[f"asin_{side}"]
    for attr in ITEM_ATTRS.values():
        tower_pairs[f"{side}_{attr}"] = asin.map(attrs_by_asin[attr])

# Shared vocabulary per attribute, exactly as in §4: an independent cast per
# column would make the same value index a different embedding row depending
# on which tower it fed.
for attr in CATEGORICAL_ITEM_ATTRS:
    q, t = f"query_{attr}", f"target_{attr}"
    level = pd.CategoricalDtype(
        sorted(set(tower_pairs[q].dropna().astype(str))
               | set(tower_pairs[t].dropna().astype(str))))
    tower_pairs[q] = tower_pairs[q].astype(level)
    tower_pairs[t] = tower_pairs[t].astype(level)

# Group each side's columns together.
SIDE_COLS = ["asin_{s}", "{s}_cat_2", "{s}_cat_3", "{s}_cat_4",
             "{s}_title_cleaned", "{s}_price", "{s}_brand_clean",
             "{s}_color", "{s}_features", "{s}_material",
             "{s}_product_type"]
tower_pairs = tower_pairs[[c.format(s=s) for s in ("query", "target")
                           for c in SIDE_COLS]]

print(f"\ntower_pairs : {tower_pairs.shape}")
for col in tower_pairs.columns:
    nn = tower_pairs[col].notna().sum()
    print(f"  {col:<24} {str(tower_pairs[col].dtype):<10} "
          f"{nn / len(tower_pairs):>6.1%} non-null")
print(f"\nmemory: {tower_pairs.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
tower_pairs.head(5)

price parsed : 515,980 of 1,134,566 items (45.5%) | median $19.01

tower_pairs : (2916635, 22)
  asin_query               object     100.0% non-null
  query_cat_2              category   100.0% non-null
  query_cat_3              category   100.0% non-null
  query_cat_4              category   100.0% non-null
  query_title_cleaned      category   100.0% non-null
  query_price              float64     75.1% non-null
  query_brand_clean        category    99.0% non-null
  query_color              category    58.5% non-null
  query_features           category    67.1% non-null
  query_material           category    76.1% non-null
  query_product_type       category    78.3% non-null
  asin_target              object     100.0% non-null
  target_cat_2             category   100.0% non-null
  target_cat_3             category   100.0% non-null
  target_cat_4             category   100.0% non-null
  target_title_cleaned     category   100.0% non-null
  target_price             float64     75

,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_brand_clean,target_color,target_features,target_material,target_product_type
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,NaN,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,simplehuman,NaN,NaN,plastic,soap pump
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,NaN,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,Angle Simple,NaN,wall mounted,stainless steel,toothbrush holder
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,NaN,NaN,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,other_brands,clear,insulated,melamine,tumbler
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,NaN,NaN,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,Tervis,orange,dishwasher safe,NaN,lid
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,NaN,NaN,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,NaN,Mr. Coffee,NaN,carafe,chrome,coffee maker


## 7. Fill the missing values

### Prices — the category median

About a quarter of rows have no price. Rather than drop them or hand the
encoder a `NaN`, each missing value takes the median price of its own category.

**The medians come from `meta_Home_and_Kitchen_filtered.csv`, not
`df_features`.** That matters more than it looks. `df_features` is filtered to
categories with an extraction schema, and the mapping's *target* side was built
from the unfiltered catalogue — so a number of reachable target paths have no
`df_features` row at all and it simply cannot price them. Of the 524 category
paths reachable in `tower_pairs`:

| | `df_features` | catalogue |
| --- | --- | --- |
| paths with no median | 60 | **11** |
| paths resting on fewer than 5 priced items | 65 | **20** |
| priced items behind the medians | 515,352 | **573,321** |

Where both can price a path they agree closely — differing by more than $1 on
28 of 464 paths. The catalogue's advantage is coverage, not different prices.

**A fallback ladder**, `cat_4 → cat_3 → cat_2 → global`, so nothing is left
unfilled: the 11 paths the catalogue cannot price at `cat_4` all resolve at
`cat_3`, and `cat_3` and `cat_2` have no gaps at all.

**One shared median table for both sides.** A category's typical price is a
property of the category, not of which tower it happens to feed.

`*_price_imputed` records which values were inferred. Filling in place without
it would hide a quarter of the column behind a plausible-looking number, and
whether a price is real is itself a signal worth keeping.

⚠️ **When you add a train/test split, fit these medians on the training rows
only.** Computed over the whole catalogue, as here, a test-period price would
leak into a training feature. Harmless today — the notebook has no split — and
a trap the moment it does.

### Categoricals — an explicit `Missing` level

Numbers get a median; categories get a name. `fill_missing_categories` gives
every missing `cat_2` / `cat_3` / `cat_4` / `brand_clean` / `color` /
`features` / `material` / `product_type` the literal value
`Missing`, the same label §3 already uses for an absent `cat_4`.

Two details this has to get right. A pandas `Categorical` **rejects a value
outside its categories**, so `Missing` must be added as a level before it can be
assigned — a plain `fillna("Missing")` raises. And both sides must come out
sharing one vocabulary: adding the level to only the side that happened to need
it would desynchronise their codes, the hazard §4 and §6 already guard against.

An explicit level is its own flag, so unlike the price there is no separate
`*_imputed` column — `Missing` in the data says exactly what happened, and the
embedding table gets one row for "no brand recorded" rather than a hole.

The three category levels arrive complete, so the function is defensive there —
"complete today" is not a property worth relying on. The gaps are real for the
rest: about 1% of rows lack a brand, and the four extracted attributes are far
sparser, `color` missing on close to half. After this cell `Missing` is the most
common single value of `query_color`, ahead of `white` and `black`. That is
honest — the model can learn that absence is informative — but half an
embedding's mass sitting on "not extracted" is a real argument for leaving
`color` out of the encoder even though the column is now present.


In [91]:
from complementary_cats_pairs import fold_cat_4, parse_category_levels

# --- Category price medians, from the unfiltered catalogue ----------------
# The category path is parsed and folded exactly as §3 folds df_features, so
# the medians key onto query_cat_2/3/4 and target_cat_2/3/4 directly.
cat_levels = parse_category_levels(meta_catalogue["category"], n_levels=4)
catalogue_prices = pd.DataFrame({
    "cat_2": cat_levels["cat_2"].astype(str),
    "cat_3": cat_levels["cat_3"].astype(str),
    "cat_4": fold_cat_4(cat_levels["cat_3"], cat_levels["cat_4"], valid_pairs),
    "price": pd.to_numeric(
        meta_catalogue["price"].astype(str).str.replace(r"[$,]", "", regex=True),
        errors="coerce"),
})

median_4 = catalogue_prices.groupby(["cat_2", "cat_3", "cat_4"], observed=True)["price"].median()
median_3 = catalogue_prices.groupby(["cat_2", "cat_3"], observed=True)["price"].median()
median_2 = catalogue_prices.groupby(["cat_2"], observed=True)["price"].median()
median_all = catalogue_prices["price"].median()

print(f"catalogue      : {catalogue_prices['price'].notna().sum():,} priced items "
      f"of {len(catalogue_prices):,}")
print(f"median tables  : {median_4.notna().sum():,} cat_4 paths | "
      f"{median_3.notna().sum()} cat_3 | {median_2.notna().sum()} cat_2 | "
      f"global ${median_all:,.2f}")


def category_median(frame, side):
    """Median price for each row's category, falling back up the hierarchy."""
    c2 = frame[f"{side}_cat_2"].astype(str)
    c3 = frame[f"{side}_cat_3"].astype(str)
    c4 = frame[f"{side}_cat_4"].astype(str)
    at_4 = pd.Series(median_4.reindex(pd.MultiIndex.from_arrays([c2, c3, c4])).to_numpy(),
                     index=frame.index)
    at_3 = pd.Series(median_3.reindex(pd.MultiIndex.from_arrays([c2, c3])).to_numpy(),
                     index=frame.index)
    at_2 = pd.Series(median_2.reindex(pd.Index(c2)).to_numpy(), index=frame.index)
    return at_4, at_3, at_2


# --- Fill both sides -------------------------------------------------------
# The original price is re-derived from `attrs_by_asin` (§6) rather than read
# off the column, so re-running this cell on an already-filled table gives the
# same answer instead of reporting nothing left to fill.
for side in ("query", "target"):
    raw = tower_pairs[f"asin_{side}"].map(attrs_by_asin["price"])
    at_4, at_3, at_2 = category_median(tower_pairs, side)
    filled = at_4.fillna(at_3).fillna(at_2).fillna(median_all)

    missing = raw.isna()
    tower_pairs[f"{side}_price"] = raw.fillna(filled)
    tower_pairs[f"{side}_price_imputed"] = missing

    rung_4 = (missing & at_4.notna()).sum()
    rung_3 = (missing & at_4.isna() & at_3.notna()).sum()
    rung_2 = (missing & at_4.isna() & at_3.isna() & at_2.notna()).sum()
    rung_g = (missing & at_4.isna() & at_3.isna() & at_2.isna()).sum()
    print(f"\n{side}: {missing.sum():,} missing of {len(tower_pairs):,} "
          f"({missing.mean():.1%}) -> filled")
    print(f"   from cat_4 median : {rung_4:>9,}")
    print(f"   from cat_3 median : {rung_3:>9,}")
    print(f"   from cat_2 median : {rung_2:>9,}")
    print(f"   from global median: {rung_g:>9,}")
    print(f"   still missing     : {tower_pairs[f'{side}_price'].isna().sum():>9,}")

# Keep each side's columns together.
SIDE_COLS = ["asin_{s}", "{s}_cat_2", "{s}_cat_3", "{s}_cat_4",
             "{s}_title_cleaned", "{s}_price", "{s}_price_imputed", "{s}_brand_clean",
             "{s}_color", "{s}_features", "{s}_material",
             "{s}_product_type"]
tower_pairs = tower_pairs[[c.format(s=s) for s in ("query", "target")
                           for c in SIDE_COLS]]

print(f"\ntower_pairs : {tower_pairs.shape}")
print(f"price now complete : "
      f"{tower_pairs['query_price'].notna().all() and tower_pairs['target_price'].notna().all()}")
tower_pairs.head(5)

catalogue      : 594,806 priced items of 1,300,540
median tables  : 697 cat_4 paths | 259 cat_3 | 14 cat_2 | global $18.72

query: 726,061 missing of 2,916,635 (24.9%) -> filled
   from cat_4 median :   726,061
   from cat_3 median :         0
   from cat_2 median :         0
   from global median:         0
   still missing     :         0

target: 721,768 missing of 2,916,635 (24.7%) -> filled
   from cat_4 median :   721,768
   from cat_3 median :         0
   from cat_2 median :         0
   from global median:         0
   still missing     :         0

tower_pairs : (2916635, 24)
price now complete : True


,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_price_imputed,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_price_imputed,target_brand_clean,target_color,target_features,target_material,target_product_type
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,NaN,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,False,simplehuman,NaN,NaN,plastic,soap pump
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,NaN,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,False,Angle Simple,NaN,wall mounted,stainless steel,toothbrush holder
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,NaN,NaN,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,False,other_brands,clear,insulated,melamine,tumbler
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,NaN,NaN,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,False,Tervis,orange,dishwasher safe,NaN,lid
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,NaN,NaN,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,21.47,True,Mr. Coffee,NaN,carafe,chrome,coffee maker


In [92]:
MISSING_LABEL = "Missing"
CATEGORICAL_ATTRS = ["cat_2", "cat_3", "cat_4", "brand_clean",
                     "color", "features", "material", "product_type"]


def fill_missing_categories(frame, attrs, label=MISSING_LABEL):
    """Give every missing categorical value an explicit `label` level.

    A pandas Categorical refuses a value that is not one of its categories, so
    the level is added before it is assigned -- `fillna` alone would raise. The
    two sides are then rebuilt on one shared vocabulary, as in §4 and §6:
    adding the level to only the side that needed it would leave the same
    category sitting on a different code in each tower.

    Idempotent -- run it again and it finds nothing to fill and changes nothing.
    """
    for attr in attrs:
        query_col, target_col = f"query_{attr}", f"target_{attr}"
        values = (set(frame[query_col].dropna().astype(str))
                  | set(frame[target_col].dropna().astype(str)))
        level = pd.CategoricalDtype(sorted(values | {label}))
        for col in (query_col, target_col):
            frame[col] = frame[col].astype(level).fillna(label)
    return frame


before = {f"{side}_{attr}": tower_pairs[f"{side}_{attr}"].isna().sum()
          for side in ("query", "target") for attr in CATEGORICAL_ATTRS}

tower_pairs = fill_missing_categories(tower_pairs, CATEGORICAL_ATTRS)

print(f"filled with {MISSING_LABEL!r}:")
for col, n in before.items():
    note = "" if n else "   (already complete)"
    print(f"  {col:<24} {n:>8,} -> {tower_pairs[col].isna().sum():,}{note}")

print("\nshared vocabulary across the two sides:")
for attr in CATEGORICAL_ATTRS:
    q, t = f"query_{attr}", f"target_{attr}"
    shared = list(tower_pairs[q].cat.categories) == list(tower_pairs[t].cat.categories)
    print(f"  {attr:<14} shared={shared}  "
          f"has {MISSING_LABEL!r}={MISSING_LABEL in tower_pairs[q].cat.categories}  "
          f"({len(tower_pairs[q].cat.categories):,} levels)")

cat_cols = [f"{s}_{a}" for s in ("query", "target") for a in CATEGORICAL_ATTRS]
print(f"\nno NaN left in any categorical: {not tower_pairs[cat_cols].isna().any().any()}")
tower_pairs.head(5)

filled with 'Missing':
  query_cat_2                     0 -> 0   (already complete)
  query_cat_3                     0 -> 0   (already complete)
  query_cat_4                     0 -> 0   (already complete)
  query_brand_clean          30,311 -> 0
  query_color              1,211,838 -> 0
  query_features            959,316 -> 0
  query_material            697,780 -> 0
  query_product_type        632,931 -> 0
  target_cat_2                    0 -> 0   (already complete)
  target_cat_3                    0 -> 0   (already complete)
  target_cat_4                    0 -> 0   (already complete)
  target_brand_clean         30,764 -> 0
  target_color             1,210,057 -> 0
  target_features           969,437 -> 0
  target_material           709,338 -> 0
  target_product_type       644,635 -> 0

shared vocabulary across the two sides:
  cat_2          shared=True  has 'Missing'=True  (8 levels)
  cat_3          shared=True  has 'Missing'=True  (70 levels)
  cat_4          shared=True 

,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_price_imputed,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_price_imputed,target_brand_clean,target_color,target_features,target_material,target_product_type
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,Missing,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,False,simplehuman,Missing,Missing,plastic,soap pump
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,Missing,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,False,Angle Simple,Missing,wall mounted,stainless steel,toothbrush holder
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,False,other_brands,clear,insulated,melamine,tumbler
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,False,Tervis,orange,dishwasher safe,Missing,lid
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,21.47,True,Mr. Coffee,Missing,carafe,chrome,coffee maker


In [96]:
print(f'Shape of the table: {tower_pairs.shape}')
print(f'Variables of the table: {tower_pairs.columns}')

Shape of the table: (2916635, 24)
Variables of the table: Index(['asin_query', 'query_cat_2', 'query_cat_3', 'query_cat_4',
       'query_title_cleaned', 'query_price', 'query_price_imputed',
       'query_brand_clean', 'query_color', 'query_features', 'query_material',
       'query_product_type', 'asin_target', 'target_cat_2', 'target_cat_3',
       'target_cat_4', 'target_title_cleaned', 'target_price',
       'target_price_imputed', 'target_brand_clean', 'target_color',
       'target_features', 'target_material', 'target_product_type'],
      dtype='object')


## 8. `target_node` — the target category as one symbol

The three target levels collapsed into a single value, e.g.
`Bath > Bathroom Accessories > Holders & Dispensers`.

This is the **target category** the paper's query tower consumes: in
Complementary-TT the query product embedding is concatenated with a target
category embedding, and that embedding needs one symbol per category to look
up, not three separate levels. Keeping the full path rather than `cat_4` alone
matters because a leaf label is not unique on its own — `Missing` and
`<cat_3>_Other` recur under many parents, so `cat_4` by itself would collapse
genuinely different categories onto one embedding row.

Built **after** §7, deliberately. Every level is complete by then, so no node
carries a `NaN` fragment or silently becomes the string `"nan"`.

Stored as `category`: a few hundred distinct paths across 2.9M rows, so the
column is integer codes against one shared vocabulary — which is what an
embedding table indexes anyway.


In [99]:
# One symbol per target category path, for the query tower's target-category
# embedding. `" > "` is cosmetic -- it only has to be readable and not appear
# inside a category name.
NODE_SEP = " / "
TARGET_LEVELS = ["target_cat_2", "target_cat_3", "target_cat_4"]

tower_pairs["target_node"] = (
    tower_pairs["target_cat_2"].astype(str) + NODE_SEP
    + tower_pairs["target_cat_3"].astype(str) + NODE_SEP
    + tower_pairs["target_cat_4"].astype(str)
).astype("category")

nodes = tower_pairs["target_node"]
print(f"target_node : {nodes.nunique():,} distinct paths over {len(nodes):,} rows")
print(f"separator   : {NODE_SEP!r} | appears inside a category name: "
      f"{any(NODE_SEP in c for lvl in TARGET_LEVELS for c in tower_pairs[lvl].cat.categories)}")
print(f"any NaN     : {nodes.isna().any()}")
print(f"memory      : {nodes.memory_usage(deep=True) / 1e6:,.1f} MB "
      f"(vs {sum(tower_pairs[c].memory_usage(deep=True) for c in TARGET_LEVELS) / 1e6:,.1f} MB "
      f"for the three levels)")

print("\nmost common target nodes:")
for path, n in nodes.value_counts().head(8).items():
    print(f"  {n:>9,}  {path}")

tower_pairs[["asin_query", "asin_target"] + TARGET_LEVELS + ["target_node"]].head(5)

target_node : 448 distinct paths over 2,916,635 rows
separator   : ' / ' | appears inside a category name: False
any NaN     : False
memory      : 5.9 MB (vs 11.7 MB for the three levels)

most common target nodes:
    112,675  Kitchen & Dining / Kitchen Utensils & Gadgets / Cooking Utensils
     80,221  Home Dcor / Window Treatments / Draperies & Curtains
     71,045  Bath / Bathroom Accessories / Shower Curtains, Hooks & Liners
     68,107  Kitchen & Dining / Kitchen Utensils & Gadgets / Measuring Tools & Scales
     66,454  Kitchen & Dining / Storage & Organization / Food Storage
     58,501  Kitchen & Dining / Kitchen Utensils & Gadgets / Graters, Peelers & Slicers
     52,160  Bedding / Decorative Pillows, Inserts & Covers / Throw Pillow Covers
     52,122  Home Dcor / Area Rugs, Runners & Pads / Area Rugs


,asin_query,asin_target,target_cat_2,target_cat_3,target_cat_4,target_node
0,0560467893,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,Bath / Bathroom Accessories / Holders & Dispen...
1,0560467893,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,Bath / Bathroom Accessories / Holders & Dispen...
2,0681795107,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Kitchen & Dining / Dining & Entertaining / Gla...
3,0681795107,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,Kitchen & Dining / Cookware / Canning
4,0681795107,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,"Kitchen & Dining / Coffee, Tea & Espresso / Co..."


In [100]:
tower_pairs.head()

,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_price_imputed,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_price_imputed,target_brand_clean,target_color,target_features,target_material,target_product_type,target_node
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,Missing,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,False,simplehuman,Missing,Missing,plastic,soap pump,Bath / Bathroom Accessories / Holders & Dispen...
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,19.37,True,WELLAND,black,floating shelf,Missing,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,False,Angle Simple,Missing,wall mounted,stainless steel,toothbrush holder,Bath / Bathroom Accessories / Holders & Dispen...
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,False,other_brands,clear,insulated,melamine,tumbler,Kitchen & Dining / Dining & Entertaining / Gla...
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,False,Tervis,orange,dishwasher safe,Missing,lid,Kitchen & Dining / Cookware / Canning
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,False,Timolino,Missing,Missing,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,21.47,True,Mr. Coffee,Missing,carafe,chrome,coffee maker,"Kitchen & Dining / Coffee, Tea & Espresso / Co..."
